# Backbone benchmarking -- Kaggle runner

Thin wrapper. All logic lives in the `bbeval` package; this notebook only
installs it, points it at the mounted datasets, and runs it.

Attach **MVTec AD** and **VisA** as datasets and enable a **GPU** accelerator.
The run finishes by writing a single ZIP of every artefact, next to
`output_root`, ready to download from the Kaggle output panel.


## 1. Fetch the code

Clones (or fast-forwards) the repository, then installs it editable. Requires
**Internet** to be switched on in the Kaggle session settings -- both the clone
and the CLIP git dependency need it.

The commit is printed so a run can be traced back to the exact source.


In [ ]:
import os
import shutil
import subprocess
import tarfile
import urllib.request

OWNER, NAME, BRANCH = "Parsagh05", "backbone_eval", "main"
REPO = f"https://github.com/{OWNER}/{NAME}.git"
ROOT = "/kaggle/working/backbone_eval"

# Without this, a blocked network makes git ask for a password and report
# "could not read Username", which says nothing about the actual cause.
os.environ["GIT_TERMINAL_PROMPT"] = "0"


def git(*args):
    """Errors and progress go straight to the cell output."""
    subprocess.run(["git", *args], check=True)


def fetch_tarball():
    """Plain HTTPS, for networks where git's smart protocol does not survive."""
    url = f"https://codeload.github.com/{OWNER}/{NAME}/tar.gz/refs/heads/{BRANCH}"
    archive = "/kaggle/working/_repo.tar.gz"
    urllib.request.urlretrieve(url, archive)
    with tarfile.open(archive) as tar:
        try:
            tar.extractall("/kaggle/working", filter="data")
        except TypeError:                      # filter= is newer than some runtimes
            tar.extractall("/kaggle/working")
    shutil.rmtree(ROOT, ignore_errors=True)
    os.replace(f"/kaggle/working/{NAME}-{BRANCH}", ROOT)
    os.remove(archive)


if os.path.isdir(os.path.join(ROOT, ".git")):
    try:
        git("-C", ROOT, "pull", "--ff-only")   # --ff-only: local edits fail loudly
    except subprocess.CalledProcessError:
        # The code is already here; a failed refresh is not worth losing it.
        print("git pull failed; using the checkout already on disk", flush=True)
else:
    shutil.rmtree(ROOT, ignore_errors=True)    # clear any half-finished attempt
    try:
        git("clone", "--depth", "1", REPO, ROOT)
    except subprocess.CalledProcessError:
        print("git clone failed; retrying over plain HTTPS", flush=True)
        fetch_tarball()

# Cloning an empty repository also succeeds and leaves nothing behind, so check
# for real content rather than failing later inside pip.
if not os.path.isfile(os.path.join(ROOT, "pyproject.toml")):
    raise RuntimeError(
        f"No pyproject.toml under {ROOT}. Most often this means Internet is "
        "switched off for this session (Notebook settings -> Internet on). "
        f"Otherwise check that {REPO} exists and has been pushed.")

if os.path.isdir(os.path.join(ROOT, ".git")):
    print("source:", subprocess.run(
        ["git", "-C", ROOT, "rev-parse", "--short", "HEAD"],
        check=True, text=True, capture_output=True).stdout.strip())
else:
    print(f"source: {BRANCH} tarball (no git metadata)")


In [ ]:
%pip install -q -e "/kaggle/working/backbone_eval[siglip2]"
# CLIP is a git dependency; skip this line for a SigLIP2-only run.
%pip install -q "git+https://github.com/openai/CLIP.git"


### Make it importable, and check what loaded

A PEP 660 editable install writes a `.pth` file, and `.pth` files are read only
when the interpreter starts -- so a fresh install is invisible to a kernel that
is already running. Pointing at the source tree avoids a kernel restart.

The backbone list is worth reading before starting a long run: a missing
optional dependency disables one backbone rather than raising, so a silent
`clip` failure here would otherwise only show up as a missing table column.


In [ ]:
import importlib
import sys

SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()

import bbeval
from bbeval.backbones import backbone_errors, backbone_names

print("bbeval", bbeval.__version__, "from", os.path.dirname(bbeval.__file__))
print("backbones:", backbone_names())
for name, reason in backbone_errors().items():
    print(f"  unavailable -- {name}: {reason}")


## 2. Settings

`siglip2_dense_readout` is the variable under test:

* `"map_token"` -- pool each patch through SigLIP2's own attention-pooling head
* `"raw"` -- leave trunk tokens unprojected; the CLIP-shaped control that
  reproduces the published chance-level localisation

`corruptions_enabled=False` keeps this to the clean backbone comparison. Turning
it on multiplies the sweep by 34 settings per category and will not fit in one
Kaggle session -- see the README.


In [ ]:
from bbeval import BackboneEvalConfig, run_evaluation

config = BackboneEvalConfig(
    mvtec_root="/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection",
    visa_root="/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922",
    output_root="/kaggle/working/results",
    weights_dir="/kaggle/working/weights",
    backbones=("clip", "siglip2"),
    siglip2_dense_readout="map_token",
    corruptions_enabled=False,
    num_workers=2,
    device="cuda",
)
print("config id:", config.fingerprint())


## 3. Smoke test

Two categories, four images each. `limit` suppresses saving, so this cannot
pollute the real artefacts -- it only proves the weights load and the shapes
line up before committing hours of GPU time.


In [ ]:
import gc

import torch
from dataclasses import replace

from bbeval.engine import load_backbones, run_shard
from bbeval.prompts import build_fixed_text

smoke = replace(config, limit=4, archive_results=False,
                categories={"mvtec": ("hazelnut",), "visa": ("candle",)})

backbones = load_backbones(smoke)
for name, backbone in backbones.items():
    text = build_fixed_text(smoke, backbone, "hazelnut")
    # save=False -> {mode: {"scores", "maps", "labels"}} plus the masks.
    outputs, masks = run_shard(smoke, backbone, {"fixed": text}, "mvtec",
                               "hazelnut", "clean", 0, save=False)
    shard = outputs["fixed"]
    print(f"{name:<10} scores {shard['scores'].shape} maps {shard['maps'].shape} "
          f"range [{shard['maps'].min():.3f}, {shard['maps'].max():.3f}] "
          f"| masks {masks.shape}")

# run_evaluation loads its own backbones, so drop these before the real run --
# two copies of CLIP-L plus SigLIP2-L will not fit beside each other on a T4.
del backbones, backbone
gc.collect()
torch.cuda.empty_cache()


## 4. Run

Fits one prompt set per source dataset, scores every category of the *other*
dataset, and writes low-resolution anomaly maps, raw scores, ground truth, the
run manifest and the metric tables.

Set `resume=True` (the default) so an interrupted session picks up where it
stopped rather than recomputing.


In [ ]:
result = run_evaluation(config)
result


## 5. Download

`result["archive"]` is the single ZIP containing artefacts, prompt checkpoints,
tables and the run manifest. It appears in the Kaggle output panel.


In [ ]:
import os
print(result["archive"], f'{os.path.getsize(result["archive"]) / 1e6:.0f} MB')
